# 25. Calibrating the simulator: coefficient fits x a team-level ingredient (simulator step 3a)

Notebook 24 found two things:

- **On its own, the simulator doesn't beat rule (b)** ("last season's points, pulled one-third toward average"). A 40/60 blend with rule (b) does.
- **It's too timid on recent seasons.** Real gaps between teams are 1.41x the simulated ones (the test-season calibration slope).

This notebook tests two fixes at once, because they may overlap. A simulator that's too timid makes good teams beat it every year, and that looks like a "team effect" too.

**Fix 1, the coefficient fit** (how many goals a WAR of edge is worth):

- **all seasons:** notebook 24's fit;
- **full-history seasons only:** target seasons from 2011-12 on. The 2009-10 and 2010-11 projections had only one or two seasons of NHL data behind them;
- **recency-weighted:** all seasons, with a season's weight halving every h seasons back.

**Fix 2, a team-level ingredient:**

- **none;**
- **a level blend:** 0.4 x simulator + 0.6 x rule (b);
- **a residual:** last season's real points minus what the simulator expected from its players, shrunk, and added to the team's strength.

**Why the residual and not the blend?** The blend fixes 60% of a team's points no matter who plays. So a sit-out or a trade would show only 40% of its effect, and those are the main things the simulator is for. The residual stays fixed when the roster changes, so a roster move keeps its full effect. Section 9 checks that with Detroit and Larkin.

**How versions are picked:** on the training seasons, then confirmed on the test seasons. Every place they disagree is said plainly.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import time
import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, project, backtest, aging_curve, group_average, current_projections, add_age, GOALS_PER_WIN
from edge_data import build_bio_table, schedule_games, nhl_code
from nhle import load_profiles
from development import (training_nhle, best_nhle, skater_features, fit_prior, apply_prior, candidates, fit_rows,
                         experience, TARGETS, AGING, SK_WEIGHTS, SK_REG, G_WEIGHTS, G_REG, PRIOR_INPUTS)
import lineups as L
import simulate as S

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

skater_war = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalie_war = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
history = history_table(skater_war, goalie_war)
skaters, goalies = history[history["group"] != "G"], history[history["group"] == "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]
SHORT_PREV = [2012, 2020]                       # short seasons: not targets, but "last season" for 2013-14 and 2021-22
SEASONS = sorted(TARGETS + SHORT_PREV)          # every season that needs lineups

# Notebooks 23 and 24's choices
CHOSEN = {"F": (0.75, "R1"), "D": (0.75, "R1"), "G": (0.5, "R1")}    # coach score: w and the rookie rule
LOOKBACK = {"F": 2, "D": 1, "G": 2}                                  # seasons looked back for usage
OWN = {"F": 0.75, "D": 0.75}                                         # minutes blend (notebook 24)
RANDOMNESS = "full"                                                  # player randomness (notebook 24, pools centered)
N_SIMS, SEED = 1000, 24

# This notebook's settings
FULL_FROM = 2011                                # full-history seasons: targets from 2011-12 on
SCORE_TRAIN = [t for t in TRAIN if t >= 2013]   # training seasons scored (see section 4)
H_GRID = [2, 4, 8]                              # recency: a season's weight halves every h seasons back
LAMBDAS = np.round(np.arange(0.1, 1.0001, 0.1), 1)   # residual shrink factors tried (0 = no ingredient)
BLEND_W = 0.4                                   # level blend: 0.4 x simulator + 0.6 x rule (b)
CARRY_PREVIEW = 0.31                            # carryover measured from notebook 24's prototype (shown to Ryan)
print("Training seasons scored:", ", ".join(label(t) for t in SCORE_TRAIN))
print("Test seasons:           ", ", ".join(label(t) for t in TEST))

Training seasons scored: 2013-14, 2014-15, 2015-16, 2016-17, 2017-18
Test seasons:            2018-19, 2021-22, 2022-23, 2023-24, 2024-25, 2025-26


## 2. Real results

The same games as notebook 24: the NHL API's saved schedules for 2008-09 to 2020-21 (no network call here) and the shot data for 2021-22 on. The late-game table and the scoring level are notebook 24's.

In [2]:
assert (DATA / "schedule_raw").exists(), "run notebook 24 first (it downloads the past schedules)"
old = schedule_games(DATA / "schedule_raw")
recent = S.games_from_shots(pd.read_parquet(DATA / "shots_2021-2025.parquet"))
games = pd.concat([old, recent], ignore_index=True)
results = S.add_results(games)
results["reg_goals"] = results["home_goals"] + results["away_goals"] - results["ot"]
real = S.standings(games)
LATE = S.late_table(recent)
reg_pg = results.groupby("season")["reg_goals"].mean()

def normal_total(seasons):
    """Normal goals per game going into a season: regulation goals per game in `seasons` minus the late goals."""
    return float(reg_pg.reindex(seasons).dropna().mean() - LATE["extra_goals"])

print(f"{len(results):,} regular-season games, {results['season'].min()}-{results['season'].max()}")

21,611 regular-season games, 2008-2025


## 3. Projections, pools and lineups for past seasons

These are exactly notebook 24's: notebook 18's projections (with its error check), pools of players with **1+ games** for the team, notebook 23's coach score, call-ups as they happened, and the minutes blend.

**New here:** the two short seasons (2012-13, 48 games; 2020-21, 56 games) get lineups too. They aren't scored, but they're the "last season" for 2013-14 and 2021-22.

In [3]:
f_train, m_train = training_nhle(careers, TRAIN_END)
nhle_train = best_nhle(careers, f_train, m_train, rule="last2")
pairs = pd.concat([candidates(history, SEASONS + [2026]),
                   skaters[skaters["season"].isin(SEASONS)][["playerId", "season", "group"]]]).drop_duplicates(["playerId", "season"])
feat = skater_features(pairs, history, bios, profiles, nhle_train)
model = fit_prior(fit_rows(history, feat, TRAIN), PRIOR_INPUTS)
cand = candidates(history, SEASONS).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
priors = cand[["playerId", "season"]].assign(prior_gar_60=apply_prior(cand, model).to_numpy())
rows = []
for t in SEASONS:
    curve = aging_curve(history, bios, exclude_season=t)[1]
    ps = project(skaters, bios, t, SK_WEIGHTS, SK_REG, AGING, curve, priors=priors, prior_reg=1500)
    pg = project(goalies, bios, t, G_WEIGHTS, G_REG, AGING, curve)
    rows.append(pd.concat([ps, pg])[["proj_war_60"]].assign(season=t).rename_axis("playerId").reset_index())
bt_proj = pd.concat(rows, ignore_index=True)
b = backtest(skaters, TEST, SK_WEIGHTS, SK_REG, 300, bios, aging=AGING, priors=priors, prior_reg=1500)
b = b.merge(feat[["playerId", "season", "age"]], on=["playerId", "season"], how="left")
k = b["age"] > 0
err = np.sqrt(np.average((b["actual"][k] - b["pred"][k]) ** 2, weights=b["toi"][k]))
assert round(err, 4) == 0.2665, err
print(f"Past-season projections rebuilt; test-season error {err:.4f} = notebook 18's")

Past-season projections rebuilt; test-season error 0.2665 = notebook 18's


In [4]:
faceoffs = L.faceoff_rates([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"])
usage = L.usage_table(skater_war, goalie_war, faceoffs)
exp = experience(history, profiles, history[["playerId", "season"]])

def make_pool(t):
    """Players with 1+ games for a team in season t, with projection and usage (notebook 24's main pool)."""
    u = usage[(usage["season"] == t) & (usage["games_played"] >= 1)]
    p = (u[["playerId", "name", "team", "position", "group", "games_played", "usage"]]
         .rename(columns={"games_played": "gp", "usage": "actual"}).reset_index(drop=True))
    p["proj_war_60"] = p["playerId"].map(bt_proj[bt_proj["season"] == t].set_index("playerId")["proj_war_60"])
    miss = p["proj_war_60"].isna() & (p["group"] != "G")
    f = p.loc[miss, ["playerId", "group"]].merge(feat[feat["season"] == t].drop(columns="group"), on="playerId", how="left")
    p.loc[miss, "proj_war_60"] = (apply_prior(f, model) / GOALS_PER_WIN).to_numpy()
    still = p["proj_war_60"].isna()
    p.loc[still, "proj_war_60"] = p.loc[still, "group"].map(group_average(history, t - 1) / GOALS_PER_WIN)
    p["usage_last"] = L.usage_by_group(usage, p, t, LOOKBACK)
    p["natural_center"] = L.natural_center(usage, p["playerId"], p["position"], t) & (p["group"] == "F")
    g = p["group"] == "G"
    p.loc[g, "actual"] = p.loc[g, "gp"] / p[g].groupby("team")["gp"].transform("sum")
    p["season"] = t
    return p

lineups = {}
for t in SEASONS:
    seasons = [t - 3, t - 2, t - 1]
    defaults, slots = L.rookie_defaults(usage, exp, seasons), L.slot_minutes(usage, seasons)
    share, totals = L.starter_share(usage, seasons)[1], L.team_minutes(usage, seasons)
    sc = L.coach_scores(make_pool(t), CHOSEN, defaults)
    lu = pd.concat([L.pick_lineup(x) for _, x in sc.groupby("team")])
    lu["team"] = [nhl_code(x, t) for x in lu["team"]]
    lu["minutes_per_game"] = L.assign_minutes(lu, L.scaled_slots(slots, totals), share, own_weight=OWN)
    lu["age"] = add_age(lu[["playerId"]].assign(season=t), bios)["age"].to_numpy()
    lu["draft_overall"] = lu["playerId"].map(bios.set_index("playerId")["draft_overall"])
    lineups[t] = lu.reset_index(drop=True)

rows = []
for t in SEASONS:
    teams, sk, gk = S.team_strength(lineups[t])
    rows.append(pd.DataFrame({"season": t, "team": teams, "skater": sk[0], "goalie": gk[0]}))
STRENGTH = pd.concat(rows, ignore_index=True)

s = STRENGTH.set_index(["season", "team"])
GAMES = results[results["season"].isin(SEASONS)].copy()
h = s.reindex(pd.MultiIndex.from_arrays([GAMES["season"], GAMES["home"]]))
a = s.reindex(pd.MultiIndex.from_arrays([GAMES["season"], GAMES["away"]]))
assert h.notna().all().all() and a.notna().all().all()          # every team in every game has a lineup
GAMES["ds"] = h["skater"].to_numpy() - a["skater"].to_numpy()
GAMES["dg"] = h["goalie"].to_numpy() - a["goalie"].to_numpy()
print(f"Lineups for {len(SEASONS)} seasons ({STRENGTH.groupby('season').size().sum()} team-seasons), "
      f"short seasons included: {', '.join(label(t) for t in SHORT_PREV)}")

Lineups for 16 seasons (493 team-seasons), short seasons included: 2012-13, 2020-21


## 4. Three ways to fit the coefficients, and how every version is scored

**The game model** is notebook 24's. A team's expected edge in normal goals is `home ice + a_s x (skater WAR per game edge) + a_g x (goalie WAR per game edge)`. Only the fit of `home`, `a_s`, `a_g` and the overtime tilt changes:

- **all seasons:** every target season before the one being predicted (notebook 24's test-season fit);
- **full-history seasons only:** the same, but only targets from 2011-12 on;
- **recency-weighted:** every target season before, with weight `0.5 ^ (seasons back / h)`. The most recent season counts 1, and one h seasons back counts half. h is picked on the training seasons.

**One change from notebook 24, and why.** Notebook 24 scored its training seasons with a fit on *all* training seasons, the scored season included (in-sample). That would hand a free edge to any version that drops or down-weights seasons. So here **every season is scored with a fit on the seasons before it only**, the way notebook 24 already scored its test seasons. As a result:

- **The training seasons scored are 2013-14 to 2017-18** (5 seasons, about 150 team-seasons). 2013-14 is the first one the full-history fit has a season behind (2011-12). 2012-13 was short.
  - That's a small sample, so small differences between versions are noise. It's said again where it matters.
- **The test seasons are notebook 24's,** with the same fit for "all seasons". So the all-seasons, no-ingredient test row must reproduce notebook 24's 14.381. It's asserted below.
- **Notebook 24's in-sample fit** on the same five training seasons is printed as a reference row.

In [5]:
def fit_for(kind, t, h=None):
    """The game model for target season t, fit only on target seasons before t (kind: all, full, recency)."""
    seasons = [s for s in TARGETS if s < t and (kind != "full" or s >= FULL_FROM)]
    g = GAMES[GAMES["season"].isin(seasons)]
    w = 0.5 ** ((t - 1 - g["season"].to_numpy()) / h) if kind == "recency" else None
    return S.fit_game_model(g, LATE, normal_total(seasons), weights=w)

def settings_for(fit, season):
    """Simulator settings for a season: the fit, plus scoring from the three seasons before it."""
    return S.game_settings(fit, LATE, normal_total([season - 3, season - 2, season - 1]))

DRAWS = {}
def draws_for(season):
    """Player randomness for a season's lineups (notebook 24's: one full draw per player per simulated season)."""
    if season not in DRAWS:
        lu = lineups[season]
        sched = results[results["season"] == season]
        gp = pd.concat([sched["home"], sched["away"]]).value_counts()
        minutes = lu["minutes_per_game"] * lu["team"].map(gp).to_numpy()
        inputs = INPUTS
        buckets = S.draw_buckets(lu["group"], lu["age"], lu["draft_overall"], inputs)
        DRAWS[season] = S.player_draws(lu["group"].to_numpy(), buckets, minutes, inputs, RANDOMNESS, N_SIMS, seed=season)
    return DRAWS[season]

def simulate(fit, season, bonus=None):
    """1,000 seasons of a real past season (its real schedule), notebook 24's seed. Per team: average and percentiles."""
    sched = results[results["season"] == season][["home", "away"]]
    table, pts = S.simulate_season(lineups[season], sched, settings_for(fit, season), N_SIMS, seed=season,
                                   draws=draws_for(season), bonus=bonus)
    return table.assign(season=season)

INPUTS = S.draw_inputs(pd.read_csv(DATA / "dev_residuals.csv"), pd.read_csv(DATA / "dev_ranges.csv"), center_pools=True)

def score(pred, d, lo80="p10", hi80="p90", lo50="p25", hi50="p75"):
    """Error (RMSE, points), average within-season correlation, calibration slope, and range coverage."""
    e = pred - d["real"]
    corr = d.assign(p=pred).groupby("season").apply(lambda x: x["p"].corr(x["real"]), include_groups=False).mean()
    pc = pred - pred.groupby(d["season"]).transform("mean")
    rc = d["real"] - d.groupby("season")["real"].transform("mean")
    out = {"error": float(np.sqrt((e ** 2).mean())), "correlation": float(corr), "calibration slope": float(np.polyfit(pc, rc, 1)[0])}
    if lo80 in d:
        out["inside 80%"] = float(((d["real"] >= d[lo80]) & (d["real"] <= d[hi80])).mean())
        out["inside 50%"] = float(((d["real"] >= d[lo50]) & (d["real"] <= d[hi50])).mean())
    return out

REAL = real[["season", "team", "gp", "points"]].rename(columns={"points": "real"})
start = time.time()
rows = []
for h in H_GRID:
    tab = pd.concat([simulate(fit_for("recency", t, h), t) for t in SCORE_TRAIN]).merge(REAL, on=["season", "team"])
    rows.append({"half-life h (seasons)": h, **score(tab["points"], tab)})
hgrid = pd.DataFrame(rows)
H = int(hgrid.sort_values("error")["half-life h (seasons)"].iloc[0])
print("Recency weighting, no team ingredient, training seasons:")
print(hgrid.round(3).to_string(index=False))
print(f"Half-life picked on the training seasons: h = {H} ({time.time() - start:.0f} seconds)")

Recency weighting, no team ingredient, training seasons:
 half-life h (seasons)  error  correlation  calibration slope  inside 80%  inside 50%
                     2 14.341        0.243              1.022       0.762       0.464
                     4 14.281        0.255              1.122       0.755       0.464
                     8 14.312        0.245              1.069       0.775       0.470
Half-life picked on the training seasons: h = 4 (5 seconds)


In [6]:
VERSIONS = {"all seasons": ("all", None), "full history (2011-12 on)": ("full", None), f"recency (h = {H})": ("recency", H)}
SCORED = SCORE_TRAIN + TEST
FITS = {(v, t): fit_for(kind, t, h) for v, (kind, h) in VERSIONS.items() for t in SCORED + [2026]}
rows = []
for v in VERSIONS:
    for t in [SCORE_TRAIN[0], TRAIN_END, TEST[0], TEST[-1], 2026]:
        f = FITS[(v, t)]
        rows.append({"version": v, "season predicted": label(t), "games fit": f["n_games"], "home": f["home"],
                     "a_s (per skater WAR)": f["a_s"], "a_g (per goalie WAR)": f["a_g"], "OT tilt": f["ot_b1"]})
print("Coefficients each version uses (normal goals per WAR per game of edge), at a few seasons:")
print(pd.DataFrame(rows).round(3).to_string(index=False))

Coefficients each version uses (normal goals per WAR per game of edge), at a few seasons:
                  version season predicted  games fit  home  a_s (per skater WAR)  a_g (per goalie WAR)  OT tilt
              all seasons          2013-14       3690 0.272                 3.246                 7.637    0.415
              all seasons          2017-18       8610 0.250                 3.935                 5.994    0.559
              all seasons          2018-19       9881 0.252                 3.632                 5.684    0.664
              all seasons          2025-26      16399 0.243                 5.930                 9.274    0.487
              all seasons          2026-27      17711 0.234                 5.925                 8.427    0.391
full history (2011-12 on)          2013-14       1230 0.278                -0.182                 8.990   -0.720
full history (2011-12 on)          2017-18       6150 0.243                 3.610                 4.763    0.611
full h

## 5. No team ingredient: each coefficient fit on its own

Notebook 24's settings: 1,000 simulated seasons per real season, its real schedule, notebook 24's seed, and the full (centered) player randomness. Nothing else changes between versions, so the differences come from the fit alone.

In [7]:
start = time.time()
NONE = {v: pd.concat([simulate(FITS[(v, t)], t) for t in SCORED]).merge(REAL, on=["season", "team"]) for v in VERSIONS}
for v in VERSIONS:
    NONE[v]["period"] = np.where(NONE[v]["season"] <= TRAIN_END, "train", "test")
nb24 = score(NONE["all seasons"].query("period == 'test'")["points"], NONE["all seasons"].query("period == 'test'"))
assert round(nb24["error"], 3) == 14.381, nb24          # = notebook 24's test row (pool 1+, full)
print(f"All seasons, no ingredient, test seasons: error {nb24['error']:.3f} = notebook 24's 14.381")
ref_fit = S.fit_game_model(GAMES[GAMES["season"].isin(TRAIN)], LATE, normal_total(TRAIN))     # notebook 24's in-sample fit
REF = pd.concat([simulate(ref_fit, t) for t in SCORE_TRAIN]).merge(REAL, on=["season", "team"])
print(f"{len(VERSIONS)} fits x {len(SCORED)} seasons x {N_SIMS:,} simulated seasons in {time.time() - start:.0f} seconds")

All seasons, no ingredient, test seasons: error 14.381 = notebook 24's 14.381
3 fits x 11 seasons x 1,000 simulated seasons in 13 seconds


## 6. The residual, and how much of it carries to the next season

**The residual for a team going into season t:**

`(last season's real points - the simulator's expected points for last season) / games played last season`

- **"Expected"** means last season's own lineups (players with 1+ games then), its real schedule, and **this version's coefficients for season t**. Last season is over by then, so nothing comes from the future.
- **Players only.** Last season's expectation doesn't include the residual from the season before, so nothing compounds.
- **Within a season,** residuals are centered (the league average is 0). Only differences between teams matter.
- **Expansion teams** (Vegas 2017-18, Seattle 2021-22) get 0. Moved franchises keep theirs (Atlanta to Winnipeg, Phoenix to Arizona, Arizona to Utah).

**The carryover check:** regress this season's residual (real minus the simulator, no ingredient) on last season's. The slope is how much carries over. The preview from notebook 24's prototype was about 31%.

In [8]:
MOVED = {("ATL", 2011): "WPG", ("PHX", 2014): "ARI", ("ARI", 2024): "UTA"}

def residuals(fit, season):
    """Real minus simulated points per game for every team in a season (centered), simulated with `fit`."""
    d = simulate(fit, season).merge(REAL, on=["season", "team"])
    r = (d["real"] - d["points"]) / d["gp"]
    return pd.Series((r - r.mean()).to_numpy(), index=d["team"])

def carried(prev, t):
    """Last season's residuals as they apply in season t (moved franchises keep theirs; new teams get 0)."""
    prev = prev.rename(index=lambda tm: MOVED.get((tm, t), tm))
    teams = sorted(lineups[t]["team"].unique()) if t in lineups else sorted(prev.index)
    return prev.reindex(teams).fillna(0.0)

start = time.time()
RESID = {(v, t): carried(residuals(FITS[(v, t)], t - 1), t) for v in VERSIONS for t in SCORED + [2026]}
print(f"Last season's residuals for {len(RESID)} version-seasons in {time.time() - start:.0f} seconds")

rows = []
for v in VERSIONS:
    d = NONE[v].copy()
    d["now"] = (d["real"] - d["points"]) / d["gp"]
    d["now"] -= d.groupby("season")["now"].transform("mean")
    d["last"] = [RESID[(v, t)].get(tm, 0.0) for t, tm in zip(d["season"], d["team"])]
    for per in ["train", "test"]:
        x = d[(d["period"] == per)]
        x = x[x["last"] != 0]                                      # expansion teams have no last season
        rows.append({"version": v, "period": per, "team-seasons": len(x), "correlation": x["now"].corr(x["last"]),
                     "carryover slope": np.polyfit(x["last"], x["now"], 1)[0],
                     "SD of residual (points per 82)": x["now"].std() * 82})
CARRY = pd.DataFrame(rows)
print("How much of last season's residual shows up again this season:")
print(CARRY.round(3).to_string(index=False))

Last season's residuals for 36 version-seasons in 12 seconds
How much of last season's residual shows up again this season:
                  version period  team-seasons  correlation  carryover slope  SD of residual (points per 82)
              all seasons  train           150        0.323            0.324                          14.283
              all seasons   test           190        0.402            0.370                          14.266
full history (2011-12 on)  train           150        0.359            0.356                          14.420
full history (2011-12 on)   test           190        0.399            0.369                          14.304
          recency (h = 4)  train           150        0.329            0.329                          14.237
          recency (h = 4)   test           190        0.368            0.341                          14.173


**Turning the residual into strength.**

`bonus (normal goals per game) = lambda x residual (points per game) / beta`

- **beta** converts goals into points: how many points per game the simulator gives for one extra normal goal per game of edge. It's measured from the simulator itself. It only sets the units: lambda is fit, so if beta is a bit off, the fit absorbs it.
- **lambda** is the shrink factor. It's picked from 0.1 to 1 by the error on the **training seasons only** (lambda = 0 is no ingredient).

The bonus is added to the team's edge in every game, and roster moves don't touch it.

In [9]:
d = NONE["all seasons"].query("period == 'train'").merge(STRENGTH, on=["season", "team"])
f = {t: FITS[("all seasons", t)] for t in SCORE_TRAIN}
d["q"] = [f[t]["a_s"] * s_ + f[t]["a_g"] * g_ for t, s_, g_ in zip(d["season"], d["skater"], d["goalie"])]
d["q"] -= d.groupby("season")["q"].transform("mean")
BETA = float(np.polyfit(d["q"], d["points"] / d["gp"], 1)[0])
print(f"beta = {BETA:.3f} points per game for one normal goal per game of edge (simulator, training seasons)")

start = time.time()
rows, RESID_RUNS = [], {}
for v in VERSIONS:
    base = score(NONE[v].query("period == 'train'")["points"], NONE[v].query("period == 'train'"))
    rows.append({"version": v, "lambda": 0.0, **base})
    for lam in LAMBDAS:
        tab = pd.concat([simulate(FITS[(v, t)], t, bonus=(lam * RESID[(v, t)] / BETA).to_dict()) for t in SCORE_TRAIN])
        tab = tab.merge(REAL, on=["season", "team"])
        rows.append({"version": v, "lambda": lam, **score(tab["points"], tab)})
LGRID = pd.DataFrame(rows)
LAMBDA = {v: float(LGRID[LGRID["version"] == v].sort_values("error")["lambda"].iloc[0]) for v in VERSIONS}
print(f"Shrink factor grid on the training seasons ({time.time() - start:.0f} seconds), error in points:")
print(LGRID.pivot(index="lambda", columns="version", values="error").round(3).to_string())
print()
for v in VERSIONS:
    c = CARRY[(CARRY["version"] == v) & (CARRY["period"] == "train")]["carryover slope"].iloc[0]
    print(f"{v:28s} lambda fit on training seasons: {LAMBDA[v]:.1f}   carryover slope, training seasons: {c:.2f}   "
          f"(notebook 24 preview: {CARRY_PREVIEW:.2f})")

beta = 0.379 points per game for one normal goal per game of edge (simulator, training seasons)
Shrink factor grid on the training seasons (46 seconds), error in points:
version  all seasons  full history (2011-12 on)  recency (h = 4)
lambda                                                          
0.0           14.325                     14.449           14.281
0.1           13.915                     14.051           13.933
0.2           13.639                     13.781           13.658
0.3           13.565                     13.631           13.611
0.4           13.559                     13.597           13.572
0.5           13.739                     13.734           13.740
0.6           13.990                     13.912           14.077
0.7           14.437                     14.222           14.499
0.8           14.957                     14.652           14.963
0.9           15.541                     15.147           15.554
1.0           16.153                     15.730   

## 7. The grid: 3 coefficient fits x 3 team ingredients

- **none:** the simulator as is.
- **level blend:** 0.4 x simulator + 0.6 x rule (b), with the weights fixed as proposed.
  - Its 80% and 50% ranges are the simulator's, moved by 0.6 x (rule (b) - the simulated average). That keeps the simulator's spread around the blended center.
  - For reference, the weight a training-season fit would pick is printed too. It isn't used.
- **residual:** the bonus from section 6, with each version's lambda.

**Columns:**

- **error:** points, RMSE.
- **correlation:** within season.
- **calibration slope:** 1 means the gaps between teams are the right size; above 1, the version is too timid.
- **inside 80% / 50%:** how often the real total lands in the simulated range (targets 80% and 50%).

In [10]:
avg_prev = (real.assign(ppg=real["points"] / real["gp"]).groupby("season")["ppg"].mean())
avg_prev.index = avg_prev.index + 1
prev = real[["season", "team"]].assign(ppg=real["points"] / real["gp"])
prev["season"] += 1
prev["team"] = [MOVED.get((tm, s), tm) for tm, s in zip(prev["team"], prev["season"])]

def rule_b(d):
    """Last season's points per game x games, pulled one-third toward the league average (new teams: average)."""
    x = d[["season", "team", "gp"]].merge(prev, on=["season", "team"], how="left")
    avg = x["season"].map(avg_prev)
    return (np.where(x["ppg"].isna(), avg, 2 / 3 * x["ppg"] + 1 / 3 * avg) * x["gp"]).to_numpy()

start = time.time()
GRID = {}
for v in VERSIONS:
    n = NONE[v].copy()
    n["rule_b"] = rule_b(n)
    GRID[(v, "none")] = n
    bl = n.copy()
    shift = (1 - BLEND_W) * (bl["rule_b"] - bl["points"])
    for c in ["points", "p10", "p25", "p75", "p90"]:
        bl[c] = bl[c] + shift
    GRID[(v, "level blend 40/60")] = bl
    rs = pd.concat([simulate(FITS[(v, t)], t, bonus=(LAMBDA[v] * RESID[(v, t)] / BETA).to_dict()) for t in SCORED])
    rs = rs.merge(REAL, on=["season", "team"])
    rs["period"] = np.where(rs["season"] <= TRAIN_END, "train", "test")
    GRID[(v, "residual")] = rs
print(f"Residual versions simulated in {time.time() - start:.0f} seconds")

rows = []
for per in ["train", "test"]:
    b_ = GRID[("all seasons", "none")].query("period == @per")
    rows.append({"period": per, "coefficient fit": "-", "team ingredient": "rule (b) alone", **score(pd.Series(b_["rule_b"].to_numpy(), index=b_.index), b_, lo80="none")})
    if per == "train":
        rows.append({"period": per, "coefficient fit": "notebook 24 in-sample fit (reference)", "team ingredient": "none", **score(REF["points"], REF)})
    for (v, ing), d in GRID.items():
        x = d[d["period"] == per]
        rows.append({"period": per, "coefficient fit": v, "team ingredient": ing, **score(x["points"], x)})
TABLE = pd.DataFrame(rows)
for per, name in [("train", "TRAINING seasons (2013-14 to 2017-18): the versions are picked here"), ("test", "TEST seasons: the confirmation")]:
    print(name)
    print(TABLE[TABLE["period"] == per].drop(columns="period").round(3).to_string(index=False))
    print()

ws = np.round(np.arange(0, 1.0001, 0.05), 2)
for v in VERSIONS:
    tr = GRID[(v, "none")].query("period == 'train'")
    w = ws[int(np.argmin([np.sqrt(((u * tr["points"] + (1 - u) * tr["rule_b"] - tr["real"]) ** 2).mean()) for u in ws]))]
    print(f"Level blend weight a training-season fit would pick, {v}: {w:.2f} x simulator (used: {BLEND_W})")

Residual versions simulated in 10 seconds
TRAINING seasons (2013-14 to 2017-18): the versions are picked here
                      coefficient fit   team ingredient  error  correlation  calibration slope  inside 80%  inside 50%
                                    -    rule (b) alone 13.875        0.421              0.624         NaN         NaN
notebook 24 in-sample fit (reference)              none 14.251        0.259              1.109       0.735       0.424
                          all seasons              none 14.325        0.241              1.037       0.768       0.497
                          all seasons level blend 40/60 13.347        0.428              0.940       0.775       0.450
                          all seasons          residual 13.559        0.408              0.813       0.768       0.483
            full history (2011-12 on)              none 14.449        0.213              1.161       0.722       0.444
            full history (2011-12 on) level blend 40/60 1

## 8. The pick: made on the training seasons, confirmed on the test seasons

- **The rule:** lowest training error wins.
- **A near tie:** if another version is within 0.1 points, the one whose calibration slope is closer to 1 and whose ranges are closer to 80% and 50% wins.
- **Then the test seasons:** where does the pick rank, which version is best there, and where do training and test disagree?

In [11]:
sims = TABLE[TABLE["coefficient fit"].isin(VERSIONS)].copy()
sims["version"] = sims["coefficient fit"] + " + " + sims["team ingredient"]
sims["shape miss"] = ((sims["calibration slope"] - 1).abs() + (sims["inside 80%"] - 0.8).abs() + (sims["inside 50%"] - 0.5).abs())
tr = sims[sims["period"] == "train"].set_index("version")
te = sims[sims["period"] == "test"].set_index("version")
near = tr[tr["error"] <= tr["error"].min() + 0.1]
PICK = near.sort_values(["shape miss", "error"]).index[0]
print(f"Lowest training error: {tr['error'].idxmin()} ({tr['error'].min():.2f})")
if len(near) > 1:
    print(f"Within 0.1 points of it: {', '.join(near.index)}; the best shape among them wins")
print(f"PICK (training seasons): {PICK}")

side = pd.DataFrame({"train error": tr["error"], "train rank": tr["error"].rank().astype(int),
                     "test error": te["error"], "test rank": te["error"].rank().astype(int),
                     "test calibration": te["calibration slope"]}).sort_values("train rank")
print("\nTraining vs. test, all nine versions:")
print(side.round(3).to_string())
rb = TABLE[(TABLE["team ingredient"] == "rule (b) alone")].set_index("period")["error"]
best_test = te["error"].idxmin()
print(f"\nOn the test seasons the pick has error {te.loc[PICK, 'error']:.2f} (rank {int(side.loc[PICK, 'test rank'])} of 9); "
      f"rule (b) alone {rb['test']:.2f}. Best on test: {best_test} ({te['error'].min():.2f}).")
print("Training and test " + ("AGREE on the pick." if best_test == PICK else "DISAGREE on the best version (said plainly in the takeaways)."))
d_ing = {ing: (tr[tr.index.str.endswith(ing)]["error"].mean(), te[te.index.str.endswith(ing)]["error"].mean())
         for ing in ["none", "level blend 40/60", "residual"]}
print("\nAverage error by team ingredient (train, test): " + "; ".join(f"{k} {a:.2f}, {b:.2f}" for k, (a, b) in d_ing.items()))
d_fit = {v: (tr[tr.index.str.startswith(v)]["error"].mean(), te[te.index.str.startswith(v)]["error"].mean()) for v in VERSIONS}
print("Average error by coefficient fit (train, test): " + "; ".join(f"{k} {a:.2f}, {b:.2f}" for k, (a, b) in d_fit.items()))

Lowest training error: recency (h = 4) + level blend 40/60 (13.32)
Within 0.1 points of it: all seasons + level blend 40/60, full history (2011-12 on) + level blend 40/60, recency (h = 4) + level blend 40/60; the best shape among them wins
PICK (training seasons): recency (h = 4) + level blend 40/60

Training vs. test, all nine versions:
                                               train error  train rank  test error  test rank  test calibration
version                                                                                                        
recency (h = 4) + level blend 40/60                 13.320           1      13.019          1             1.127
full history (2011-12 on) + level blend 40/60       13.330           2      13.071          3             1.169
all seasons + level blend 40/60                     13.347           3      13.063          2             1.173
all seasons + residual                              13.559           4      13.233          4       

## 9. Does a version mute roster moves? Detroit 2026-27 with and without Larkin

- **The 2026-27 setup:** notebook 24's opening-night lineups (call-up tiers, the minutes blend), with each version's coefficients fit on every season through 2025-26 (its own way), on notebook 24's balanced 84-game schedule. The real schedule comes in a later step.
- **The run:** 10,000 simulated seasons, with the same random numbers with and without Larkin, so the difference is his absence alone.
- **What to look for:**
  - **No ingredient and residual:** the full simulated effect. The residual is a team number, and it stays when Larkin sits.
  - **Level blend:** only 40% of it, because 60% of the points come from last season's total.

In [12]:
RECENT = [2023, 2024, 2025]
players = L.load_players(DATA, usage, current_projections(DATA), LOOKBACK)
DEFAULTS = L.rookie_defaults(usage, exp, RECENT)
slots26 = L.scaled_slots(L.slot_minutes(usage, RECENT), L.team_minutes(usage, RECENT))
SHARE = L.starter_share(usage, RECENT)[1]
scored26 = L.coach_scores(players[players["in_pool"]], CHOSEN, DEFAULTS)
LARKIN = int(players.loc[players["name"] == "Dylan Larkin", "playerId"].iloc[0])

def lineup26(sit_out=()):
    lu = L.build_lineups(scored26, slots26, SHARE, sit_out=sit_out, tiered=True, own_weight=OWN)
    lu = lu.merge(current_projections(DATA)[["playerId", "age"]], on="playerId", how="left")
    lu["draft_overall"] = lu["playerId"].map(bios.set_index("playerId")["draft_overall"])
    return lu

with_l = lineup26()
without = lineup26([LARKIN])
gone = with_l[with_l["playerId"] == LARKIN].assign(dressed=False, minutes_per_game=0.0)
without = pd.concat([without, gone]).set_index("playerId").loc[with_l["playerId"]].reset_index()   # same rows, same order
assert (without["playerId"].to_numpy() == with_l["playerId"].to_numpy()).all()
sched26 = S.balanced_schedule(sorted(with_l["team"].unique()), L.SEASON_GAMES_2026, seed=SEED)
buckets26 = S.draw_buckets(with_l["group"], with_l["age"], with_l["draft_overall"], INPUTS)
N26 = 10000

def run26(lu, sett, bonus=None):
    dr = S.player_draws(lu["group"].to_numpy(), buckets26, lu["minutes_per_game"] * 84, INPUTS, RANDOMNESS, N26, seed=SEED)
    return S.simulate_season(lu, sched26, sett, N26, seed=SEED, draws=dr, bonus=bonus)[0].set_index("team")["points"]

r25 = real[real["season"] == 2025].assign(ppg=lambda d: d["points"] / d["gp"])
det_b = (2 / 3 * r25.set_index("team").loc["DET", "ppg"] + 1 / 3 * r25["ppg"].mean()) * 84
start = time.time()
rows, BONUS26 = [], {}
for v in VERSIONS:
    sett = S.game_settings(FITS[(v, 2026)], LATE, normal_total(RECENT))
    BONUS26[v] = (LAMBDA[v] * RESID[(v, 2026)] / BETA).to_dict()
    a0, a1 = run26(with_l, sett)["DET"], run26(without, sett)["DET"]
    r0, r1 = run26(with_l, sett, BONUS26[v])["DET"], run26(without, sett, BONUS26[v])["DET"]
    b0, b1 = BLEND_W * a0 + (1 - BLEND_W) * det_b, BLEND_W * a1 + (1 - BLEND_W) * det_b
    for ing, x0, x1 in [("none", a0, a1), ("level blend 40/60", b0, b1), ("residual", r0, r1)]:
        rows.append({"coefficient fit": v, "team ingredient": ing, "DET with Larkin": x0, "without Larkin": x1,
                     "change (points)": x1 - x0, "share of the full effect": (x1 - x0) / (a1 - a0)})
LARK = pd.DataFrame(rows)
print(f"Detroit 2026-27, average of {N26:,} seasons ({time.time() - start:.0f} seconds). Rule (b) for Detroit: {det_b:.1f} points")
print(LARK.round(2).to_string(index=False))
print(f"\nDetroit's residual from 2025-26 (points per game, centered): "
      + ", ".join(f"{v} {RESID[(v, 2026)]['DET']:+.3f}" for v in VERSIONS))

Detroit 2026-27, average of 10,000 seasons (49 seconds). Rule (b) for Detroit: 94.3 points
          coefficient fit   team ingredient  DET with Larkin  without Larkin  change (points)  share of the full effect
              all seasons              none            88.42           84.70            -3.72                      1.00
              all seasons level blend 40/60            91.95           90.47            -1.49                      0.40
              all seasons          residual            91.59           88.01            -3.58                      0.96
full history (2011-12 on)              none            88.20           84.67            -3.53                      1.00
full history (2011-12 on) level blend 40/60            91.87           90.45            -1.41                      0.40
full history (2011-12 on)          residual            91.47           87.72            -3.75                      1.06
          recency (h = 4)              none            87.72         

In [13]:
pv, ping = PICK.split(" + ")
print(f"The pick ({PICK}) for 2026-27:")
if ping == "residual":
    r = RESID[(pv, 2026)].sort_values()
    print(f"  lambda {LAMBDA[pv]:.1f}; the largest residuals carried from 2025-26 (points per 84 games before the shrink):")
    print("  top:    " + ", ".join(f"{t} {x * 84:+.1f}" for t, x in r[::-1].head(5).items()))
    print("  bottom: " + ", ".join(f"{t} {x * 84:+.1f}" for t, x in r.head(5).items()))
x = LARK[(LARK["coefficient fit"] == pv) & (LARK["team ingredient"] == ping)].iloc[0]
print(f"  Detroit: {x['DET with Larkin']:.1f} points with Larkin, {x['without Larkin']:.1f} without ({x['change (points)']:+.2f})")

# The trade-off in one place: the pick vs. the best version that keeps a roster move's full effect (90%+ of it)
full = LARK[LARK["share of the full effect"] >= 0.9]
full_v = (full["coefficient fit"] + " + " + full["team ingredient"]).tolist()
KEEP = tr.loc[full_v, "error"].idxmin()
print(f"\nBest version on the training seasons that keeps roster moves at full effect: {KEEP}")
print(f"  error, training: {tr.loc[KEEP, 'error']:.2f} vs. the pick's {tr.loc[PICK, 'error']:.2f}; "
      f"test: {te.loc[KEEP, 'error']:.2f} vs. {te.loc[PICK, 'error']:.2f}; rule (b) alone {rb['train']:.2f} and {rb['test']:.2f}")
per_season = lambda name: GRID[tuple(name.split(" + "))].groupby("season").apply(
    lambda d: np.sqrt(((d["points"] - d["real"]) ** 2).mean()), include_groups=False)
ps = pd.DataFrame({PICK: per_season(PICK), KEEP: per_season(KEEP)})
ps["lower error"] = np.where(ps[PICK] < ps[KEEP], "pick", "keeps roster moves")
ps.index = ps.index.map(label)
print("\nError by season:")
print(ps.round(2).to_string())
print(f"The pick has the lower error in {(ps['lower error'] == 'pick').sum()} of {len(ps)} seasons.")

The pick (recency (h = 4) + level blend 40/60) for 2026-27:
  Detroit: 91.7 points with Larkin, 90.1 without (-1.59)

Best version on the training seasons that keeps roster moves at full effect: all seasons + residual
  error, training: 13.56 vs. the pick's 13.32; test: 13.23 vs. 13.02; rule (b) alone 13.87 and 13.56

Error by season:
         recency (h = 4) + level blend 40/60  all seasons + residual         lower error
season                                                                                  
2013-14                                12.85                   12.86                pick
2014-15                                13.35                   13.48                pick
2015-16                                10.65                   10.91                pick
2016-17                                13.80                   13.89                pick
2017-18                                15.43                   16.06                pick
2018-19                                1

## 10. The step 3 default: all seasons + residual (Ryan's choice)

The training-season rule picked recency + the level blend. **Step 3 uses all seasons + residual instead**, for two reasons:

1. **The accuracy gap is small.** 13.23 vs. 13.02 on the test seasons is within noise, and both beat rule (b) (13.56).
2. **The level blend mutes roster moves to 40%.** Sit-outs and trades are what the simulator is for. The residual keeps their full effect.

**The settings for 2026-27:**

- **Coefficients:** fit on every target season through 2025-26, equal weights (notebook 24's final fit).
- **Bonus for each team:** 0.4 x its 2025-26 residual, turned into normal goals per game.

In [14]:
STEP3 = ("all seasons", "residual")
v = STEP3[0]
f = FITS[(v, 2026)]
print(f"Step 3 default: {v} + residual. lambda {LAMBDA[v]:.1f} (fit on training seasons), beta {BETA:.3f}")
print(f"Coefficients for 2026-27 ({f['n_games']:,} games): home {f['home']:.3f}, a_s {f['a_s']:.3f}, a_g {f['a_g']:.3f}, "
      f"OT tilt {f['ot_b1']:.3f}")
bonus = pd.DataFrame({"2025-26 residual (points per 84)": RESID[(v, 2026)] * 84,
                      "after the shrink (points per 84)": LAMBDA[v] * RESID[(v, 2026)] * 84,
                      "bonus (normal goals per game)": pd.Series(BONUS26[v])}).sort_values("bonus (normal goals per game)", ascending=False)
print("\nEach team's 2026-27 bonus (residual from 2025-26), best to worst:")
print(bonus.round(3).to_string())
x = LARK[(LARK["coefficient fit"] == v) & (LARK["team ingredient"] == "residual")].iloc[0]
print(f"\nDetroit: {x['DET with Larkin']:.1f} points with Larkin, {x['without Larkin']:.1f} without "
      f"({x['change (points)']:+.2f}, {x['share of the full effect']:.0%} of the full effect)")

Step 3 default: all seasons + residual. lambda 0.4 (fit on training seasons), beta 0.379
Coefficients for 2026-27 (17,711 games): home 0.234, a_s 5.925, a_g 8.427, OT tilt 0.391

Each team's 2026-27 bonus (residual from 2025-26), best to worst:
     2025-26 residual (points per 84)  after the shrink (points per 84)  bonus (normal goals per game)
COL                            25.453                            10.181                          0.320
CAR                            16.288                             6.515                          0.205
MIN                            15.328                             6.131                          0.193
PHI                            15.164                             6.066                          0.190
BUF                            13.062                             5.225                          0.164
DAL                            10.921                             4.368                          0.137
BOS                            10.

## Takeaways
- **The step 3 default: all seasons + residual (Ryan's choice).** It's not the version the training-season rule picked, and there are two reasons:
  1. **The accuracy gap is within noise.** It's 13.23 vs. 13.02 points on the test seasons (13.56 vs. 13.32 on training), and both beat rule (b): 13.56 on test, 13.87 on training.
  2. **The level blend mutes roster moves to 40%.** Sit-outs and trades are what the simulator is for. The residual keeps their full effect.
- **What the rule picked:** recency-weighted fit (half-life 4) + the level blend 40/60.
  - **Its error:** 13.32 on training, 13.02 on test. Training and test agree, and all three blends beat all three residual versions in both periods.
  - **Season by season,** it has the lower error in 9 of 11 seasons. Most of those gaps are a tenth of a point or two; the big one is 2024-25 (12.35 vs. 13.77).
- **Detroit without Larkin in 2026-27:**
  - Level blend: -1.4 to -1.6 points, exactly 40% of the effect.
  - No ingredient: -3.5 to -4.0 points.
  - Residual: -3.6 to -4.0 points, 96% to 106% of the effect; the small wobble is simulation noise.
  - With the step 3 default, -3.6 points.
- **The residual carries over about one-third, and the shrink factor agrees:**
  - **Carryover slope** (this season's residual on last season's): 0.32 on training seasons and 0.37 on test with the all-seasons fit (0.33 to 0.37 with the others). The preview from notebook 24's prototype was 0.31.
  - **Shrink factor lambda** (fit on training seasons only): 0.4 for all three fits, a bit above the carryover slope. 0.3 is almost as good (13.565 vs. 13.559).
- **The residual fixes the simulator's timidity.** The test calibration slope goes from 1.41 to 1.00 (all seasons + residual), so the gaps between teams are the right size. The blend gets to 1.13 to 1.17.
  - **Its ranges are a little wide:** 57% of real totals land in the 50% range on test, and 78% in the 80% range.
- **The coefficient fit barely matters once an ingredient is in:**
  - **Average errors (train, test):** all seasons 13.74, 13.56; full history 13.79, 13.57; recency 13.72, 13.55.
  - **Full-history only is the worst.** Dropping 2009-10 and 2010-11 doesn't help, and early on it leaves too few games: the 2013-14 fit, on 2011-12 alone, gave skater WAR a coefficient of -0.18.
  - **Recency weighting helps the simulator on its own:** test calibration goes from 1.41 to 1.15, and error from 14.38 to 14.29. With the residual it adds nothing (13.34 vs. 13.23).
  - **Half-life 4 won on training,** narrowly (14.28; 14.34 with h = 2, 14.31 with h = 8).
- **Said plainly:**
  - **Small samples:** only five training seasons (150 team-seasons) are scored, so differences of a tenth or two are close to noise. The blend's edge over the residual does show up in both periods, though.
  - **Scoring changed from notebook 24:** every season, training included, is now scored on fits from earlier seasons only. Notebook 24's in-sample fit scores 14.25 on the same five training seasons, vs. 14.33 here.
  - **Notebook 24's leaks still apply:** traded players count for their final team, and the pool knows who played at all.
- **Detroit beat its projection last season:** +0.10 points per game in 2025-26, about +8 points per 84 games before the shrink and +3.3 after. With the residual, Detroit goes from 88.4 to 91.6 points (all seasons).
- **The 2026-27 bonuses, from 2025-26:**
  - **Biggest:** Colorado +10.2 points per 84 after the shrink (+25.5 before), Carolina +6.5, Minnesota +6.1.
  - **Smallest:** Vancouver -13.4 (-33.6 before), Toronto -10.8, Chicago -6.9.
  - **Said plainly:** the residual also soaks up last season's injuries and luck, because the pool keeps injured players in at full health. That part mostly doesn't repeat, which is why only 0.4 is kept.
- **Checks:**
  - The all-seasons, no-ingredient test row reproduces notebook 24's 14.381.
  - Notebook 18's 0.2665 reproduces.
  - Notebook 24 reproduces exactly with the new `simulate.py` (its options are off by default).
- **Next (step 3):** 2026-27 with the default (all seasons + residual, lambda 0.4): the real schedule, trades, sit-outs and start dates.